# Conferência de idempotência

Prova de que o Job pode rodar mais de uma vez sem duplicar nem alterar os dados. Depois de cada execução do Job, este notebook registra a quantidade de linhas e um checksum do conteúdo de cada tabela e view da Bronze, Silver e Gold. A última célula compara a execução 1 com a 2.

- O checksum é a soma do `xxhash64` de todas as linhas. Ele não depende da ordem das linhas e muda se qualquer valor mudar.
- Ficam de fora as colunas que registram o momento da carga (`ingestion_datetime`, `ingestion_datetime_horarios` e `data_execucao`), porque elas mudam a cada execução.
- O resultado fica em `vcredit.auditoria.conferencia_idempotencia`, fora das camadas conferidas.

In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("execucao", "1", "Execução do Job")
execucao = int(dbutils.widgets.get("execucao"))

catalogo = "vcredit"
camadas = ["bronze", "silver", "gold"]
colunas_de_carga = {"ingestion_datetime", "ingestion_datetime_horarios", "data_execucao"}
destino = f"{catalogo}.auditoria.conferencia_idempotencia"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.auditoria")

DataFrame[]

In [0]:
registros = []
for camada in camadas:
    for linha in spark.sql(f"SHOW TABLES IN {catalogo}.{camada}").collect():
        df = spark.table(f"{catalogo}.{camada}.{linha.tableName}")
        colunas = [c for c in df.columns if c not in colunas_de_carga]
        resultado = df.agg(
            F.count(F.lit(1)).alias("linhas"),
            F.sum(F.xxhash64(*colunas).cast("decimal(38,0)")).cast("string").alias("checksum"),
        ).first()
        registros.append((execucao, camada, linha.tableName, resultado.linhas, resultado.checksum))

conferencia = (
    spark.createDataFrame(registros, "execucao int, camada string, tabela string, linhas long, checksum string")
    .withColumn("data_conferencia", F.current_timestamp())
)

# Rodar de novo a mesma execução substitui o registro anterior em vez de duplicar
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {destino}
    (execucao INT, camada STRING, tabela STRING, linhas BIGINT, checksum STRING, data_conferencia TIMESTAMP)
""")
spark.sql(f"DELETE FROM {destino} WHERE execucao = {execucao}")
conferencia.write.mode("append").saveAsTable(destino)

display(conferencia.orderBy("camada", "tabela"))

execucao,camada,tabela,linhas,checksum,data_conferencia
2,bronze,tb_atendentes,25,-13546558022675672769,2026-10-28T20:16:53.191Z
2,bronze,tb_canais,4,5507556571327933423,2026-10-28T20:16:53.191Z
2,bronze,tb_chamados,134474,-4349272895256497078958,2026-10-28T20:16:53.191Z
2,bronze,tb_chamados_hora,131838,-606556843821568796609,2026-10-28T20:16:53.191Z
2,bronze,tb_clientes,47705,-186821228812366623563,2026-10-28T20:16:53.191Z
2,bronze,tb_custos,131069,-848613343033963990208,2026-10-28T20:16:53.191Z
2,bronze,tb_motivos,14,-23588495138475008270,2026-10-28T20:16:53.191Z
2,bronze,tb_pesquisa_satisfacao,81021,-3532118576950774768199,2026-10-28T20:16:53.191Z
2,gold,dim_atendente,27,10341492398377133593,2026-10-28T20:16:53.191Z
2,gold,dim_calendario,730,-1409231843173164321,2026-10-28T20:16:53.191Z


In [0]:
conferencias = spark.table(destino)
execucao_1 = conferencias.filter("execucao = 1").select("camada", "tabela", F.col("linhas").alias("linhas_1"), F.col("checksum").alias("checksum_1"))
execucao_2 = conferencias.filter("execucao = 2").select("camada", "tabela", F.col("linhas").alias("linhas_2"), F.col("checksum").alias("checksum_2"))

comparacao = (
    execucao_1.join(execucao_2, ["camada", "tabela"], "full")
    .withColumn("identica", F.coalesce(
        (F.col("linhas_1") == F.col("linhas_2")) & (F.col("checksum_1") == F.col("checksum_2")),
        F.lit(False),
    ))
    .orderBy("camada", "tabela")
)

display(comparacao)
identicas = comparacao.filter("identica").count()
print(f"{identicas} de {comparacao.count()} tabelas e views idênticas entre as execuções 1 e 2.")

camada,tabela,linhas_1,checksum_1,linhas_2,checksum_2,identica
bronze,tb_atendentes,25,-13546558022675672769,25,-13546558022675672769,true
bronze,tb_canais,4,5507556571327933423,4,5507556571327933423,true
bronze,tb_chamados,134474,-4349272895256497078958,134474,-4349272895256497078958,true
bronze,tb_chamados_hora,131838,-606556843821568796609,131838,-606556843821568796609,true
bronze,tb_clientes,47705,-186821228812366623563,47705,-186821228812366623563,true
bronze,tb_custos,131069,-848613343033963990208,131069,-848613343033963990208,true
bronze,tb_motivos,14,-23588495138475008270,14,-23588495138475008270,true
bronze,tb_pesquisa_satisfacao,81021,-3532118576950774768199,81021,-3532118576950774768199,true
gold,dim_atendente,27,10341492398377133593,27,10341492398377133593,true
gold,dim_calendario,730,-1409231843173164321,730,-1409231843173164321,true


34 de 34 tabelas e views idênticas entre as execuções 1 e 2.
